# Robot kurier: pierwsza misja z ROS 2 🐢

Zrobimy małego robota, który rysuje trasę, odbiera polecenia i zgłasza dostarczenie paczki.
Nie musisz znać ROS. Wystarczy znajomość podstaw Pythona: zmiennych, funkcji i instrukcji `if`.
Wszystkie roboty w tej części są **symulowane**.

Po ukończeniu ćwiczeń 1–5 będziesz umieć połączyć dwa programy przez topic, odczytać dane robota
w callbacku oraz napisać klienta i serwer usługi. Po drodze: rysowanie, radio,
polowanie na punkty i punkt odbioru potwierdzający dostawę.

**Jak pracujemy:** najpierw przewidź wynik, potem uruchom kod, a na końcu zmień jeden element.
Błąd jest wskazówką. Instrukcje zawierają oczekiwane wyniki i podpowiedzi.

## 1. Przygotuj bazę

Na **swoim komputerze** zainstaluj [Dockera](https://docs.docker.com/get-started/get-docker/),
a następnie uruchom w terminalu:

```bash
git clone --branch master https://github.com/AdoHaha/ros_fun.git
cd ros_fun
docker compose up
```

Przy pierwszym uruchomieniu Docker pobierze duży, gotowy obraz z ROS 2 Jazzy.
Poczekaj, aż w logach pojawi się adres Jupytera.
Jeśli masz już kopię repozytorium, uruchom `docker compose up` w jego katalogu; nie klonuj go ponownie.

| Okno | Adres na Twoim komputerze | Do czego służy |
| --- | --- | --- |
| Notebook | http://localhost:8888 | Kod i instrukcje; jeśli pyta o token, skopiuj adres z logów |
| Pulpit robota | http://localhost:6080 | Kliknij **Connect**; tutaj zobaczysz turtlesim, Gazebo i terminale |

Notebooki działają **wewnątrz kontenera**, w którym ROS jest już zainstalowany. Polecenia `ros2 ...`
uruchamiaj w terminalu pulpitu kontenera lub w terminalu Jupytera, nie na swoim komputerze poza kontenerem.
Nie zamykaj terminala z `docker compose up` podczas warsztatu.

## 2. Jak używać notebooka

Kliknij komórkę i naciśnij **Shift+Enter**. Kod wykonuj od góry do dołu.
`In [*]` oznacza, że kod nadal się wykonuje. Przycisk ■ przerywa jego wykonywanie.
**Kernel → Restart** uruchamia kernel ponownie i usuwa wszystkie zmienne notebooka. Restart nie zawsze zamyka osobno uruchomione programy.
Na końcu każdej misji jest komórka sprzątająca.

Każdy notebook ma własny kernel, czyli osobny proces Pythona. Zmienna z ćwiczenia 3
nie istnieje w ćwiczeniu 4, ale ich węzły mogą komunikować się przez ROS, gdy oba notebooki działają.
Nie uruchamiaj dwóch sterowników tego samego robota jednocześnie.

Sprawdźmy środowisko. Ta komórka niczego nie instaluje ani nie uruchamia robota.

In [ ]:
import os
import shutil
from importlib.util import find_spec

checks = {
    'ROS 2 Jazzy': os.environ.get('ROS_DISTRO') == 'jazzy',
    'polecenie ros2': shutil.which('ros2') is not None,
    'Python → ROS (rclpy)': find_spec('rclpy') is not None,
    'symulator turtlesim': find_spec('turtlesim') is not None,
    'przyciski w Jupyterze': find_spec('ipywidgets') is not None,
}
for name, ready in checks.items():
    print(f"{'✓' if ready else '✗'} {name}")
if not all(checks.values()):
    print('Otwórz ten notebook w Jupyterze z kontenera warsztatowego; sprawdź logi docker compose.')
else:
    print('Baza gotowa! W ćwiczeniu 2 pojawi się pierwszy robot.')

## 3. Co naprawdę robi ROS?

Robot ma kilka zadań naraz: odbierać komendy, mierzyć położenie, wykrywać przeszkody
czy wyświetlać wynik gry. Zamiast jednego wielkiego programu uruchamiamy współpracujące **węzły (ang. nodes)**.
Węzeł to uczestnik sieci ROS. Jeden proces Pythona może zawierać kilka węzłów.
ROS nie zastępuje systemu Ubuntu; daje programom interfejsy komunikacji i narzędzia.

```text
notebook: /kurier ── /turtle1/cmd_vel ──► /turtlesim
                       Twist              │
notebook: /tablica ◄── /turtle1/pose ──────┘
                       Pose
```

**Nazwa topicu** wskazuje kanał, którym wysyłamy dane. **Typ wiadomości** mówi, jakie
pola musi zawierać wiadomość. `/turtle1/cmd_vel` przenosi prędkość; `/turtle1/pose` przenosi
pozycję i kierunek. Programy mogą być napisane w różnych językach i działać na różnych komputerach.

## 4. Wybierz sposób rozmowy

| Mechanizm | Przykład kuriera | Co otrzymujemy? |
| --- | --- | --- |
| **Topic**: publisher → subscriber | „Jedź z prędkością 1”; „jestem w x=4, y=6” | Strumień wiadomości; może go odbierać wiele węzłów; bez osobnej odpowiedzi na każdą wiadomość |
| **Service**: klient → serwer → klient | „Zmień kolor pisaka”; „czy dostawa została przyjęta?” | Jedną odpowiedź na konkretne żądanie |
| **Action** | „Dojedź do kuchni” | Przyjęcie celu, postęp, wynik i możliwość anulowania; późniejsza część kursu |
| **Parameter** | „Jaki próg odległości ustawiono?” | Konfigurację węzła; późniejsza część kursu |

**Rozgrzewka w parach:** wybierz topic, service lub action:

1. Robot podaje odległość od ściany dziesięć razy na sekundę.
2. Prosisz o wyczyszczenie rysunku i czekasz na potwierdzenie.
3. Zlecasz kilkuminutowy przejazd i chcesz widzieć postęp.

<details><summary>Sprawdź rozumowanie</summary>
1: topic, bo dane płyną stale. 2: service, bo to krótka operacja z odpowiedzią.
3: action, bo zadanie trwa dłużej i chcemy śledzić jego postęp.
Nie wybieramy mechanizmu według języka programowania.
</details>

## 5. Nasza mapa misji

| Misja | Co zbudujesz | Orientacyjnie |
| --- | --- | --- |
| [2. Rusz robota](2.%20some%20start%20demos%20-%20let's%20move%20robots!.ipynb) | Rysunek i pierwszą trasę pokonaną za pomocą klawiatury | 15–20 min |
| [3. Publisher](3.%20ROS%20Topic%20-%20Publisher.ipynb) | Radio kuriera i panel sterowania w Pythonie | 25–35 min |
| [4. Subscriber](4.%20ROS%20Topic%20-%20Subscriber.ipynb) | Licznik punktów za odwiedzone miejsca | 25–35 min |
| [5. Service](5.%20ROS%20Service.ipynb) | Kolorowy pisak i punkt odbioru paczek | 25–35 min |

Wykonaj podstawową część każdej misji; bonusy są dla chętnych. Turtlesim uruchamia się szybko i pozwala skupić się
na ROS. **Te same mechanizmy** wykorzystamy w dodatkach do sterowania kołowym TurtleBotem z kamerą i laserem.

Dalej: [6. Nav2](6.%20Navigate%20-%20nav2.ipynb), [7. Action](7.%20ROS%20Action.ipynb),
[8. Parameters](8.%20Parameters.ipynb). Ćwiczenia 9–14 o drzewach zachowań i planowaniu
należą do osobnego, bardziej zaawansowanego kursu.

[Zaczynamy: 2. Rusz robota →](2.%20some%20start%20demos%20-%20let's%20move%20robots!.ipynb)